# Extract Skills from Job Descriptions (JD) - A100 Optimized

This notebook processes multiple parquet files optimized for A100 GPU servers.

## Key Features:
- ✅ Fixed import path for local skillner module
- ✅ Parallel processing with multiple CPU cores
- ✅ Optimized for high-memory servers (256GB+)
- ✅ Progress tracking with tqdm
- ✅ Incremental saving of results
- ✅ Resume capability if process is interrupted

## A100 Server Configuration:
- Recommended: BATCH_SIZE = 50 (process 50 files at once)
- Recommended: ROWS_PER_CHUNK = 10000 (10k rows per chunk)
- Recommended: N_WORKERS = 16-32 (parallel processing)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc
import sys
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Fix import path for local skillner module
# Add parent directory to Python path to import skillner
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))
    print(f"Added {parent_dir} to Python path")

# Now import skillner
from skillner import Pipeline
print("✓ SkillNER imported successfully!")

## Configuration - A100 Optimized

In [ ]:
# ============================================================================
# CONFIGURATION - A100 Server Optimized
# ============================================================================

# Input/Output paths
INPUT_DIR = Path("../JD")  # Directory containing parquet files
OUTPUT_DIR = Path("./output/extracted_skills")  # Output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# A100 SERVER CONFIGURATION (High Memory)
# ============================================================================
# Assuming 256GB+ RAM available

BATCH_SIZE = 50  # Process 50 files at once (vs default 10)
ROWS_PER_CHUNK = 10000  # Process 10k rows per chunk (vs default 1000)

# Parallel processing
USE_PARALLEL = True  # Enable parallel processing
N_WORKERS = 16  # Number of parallel workers (adjust based on CPU cores)
                # A100 servers typically have 32-64+ CPU cores
                # Recommended: Use 50-75% of available cores

# ============================================================================
# Alternative configurations based on your data size:
# ============================================================================
# 
# CONSERVATIVE (if files are very large):
# BATCH_SIZE = 20
# ROWS_PER_CHUNK = 5000
# N_WORKERS = 8
#
# AGGRESSIVE (if you have 512GB+ RAM):
# BATCH_SIZE = 100
# ROWS_PER_CHUNK = 20000
# N_WORKERS = 32
# ============================================================================

# Column names
TEXT_COLUMN = 'job_description'  # Name of the column containing text
ID_COLUMN = 'job_id'  # Name of the ID column (optional)

# Resume capability
RESUME_FROM_CHECKPOINT = True  # Set to True to skip already processed files
CHECKPOINT_FILE = OUTPUT_DIR / "processed_files.txt"  # Tracks processed files

print("="*70)
print("A100 OPTIMIZED CONFIGURATION")
print("="*70)
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE} files")
print(f"Chunk size: {ROWS_PER_CHUNK} rows")
print(f"Parallel processing: {USE_PARALLEL}")
if USE_PARALLEL:
    print(f"Number of workers: {N_WORKERS}")
print("="*70)

## System Information

In [ ]:
import psutil
import os

# Check system resources
cpu_count = psutil.cpu_count(logical=True)
cpu_count_physical = psutil.cpu_count(logical=False)
vm = psutil.virtual_memory()

print("System Information:")
print(f"  CPU cores (physical): {cpu_count_physical}")
print(f"  CPU cores (logical): {cpu_count}")
print(f"  Total RAM: {vm.total / (1024**3):.2f} GB")
print(f"  Available RAM: {vm.available / (1024**3):.2f} GB")
print(f"  RAM usage: {vm.percent}%")

# Recommendation
if vm.total / (1024**3) > 200:
    print("\n✓ High memory system detected! Configuration is optimized for your hardware.")
    recommended_workers = min(cpu_count_physical * 2, 32)
    if N_WORKERS < recommended_workers:
        print(f"  💡 Consider increasing N_WORKERS to {recommended_workers} for better performance")
elif vm.total / (1024**3) > 100:
    print("\n⚠️  Medium memory system. Consider reducing BATCH_SIZE to 30")
else:
    print("\n⚠️  Low memory system. Reduce BATCH_SIZE to 10 and ROWS_PER_CHUNK to 1000")

## Helper Functions

In [ ]:
def get_processed_files():
    """Get set of already processed files from checkpoint"""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return set(line.strip() for line in f)
    return set()

def mark_file_as_processed(filename):
    """Add filename to checkpoint file"""
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{filename}\n")

def process_text_batch(texts, nlp):
    """Extract skills from a batch of texts"""
    results = []
    for text in texts:
        if pd.isna(text) or text == '':
            results.append({'skills': [], 'num_skills': 0})
        else:
            try:
                annotations = nlp(text)
                skills = [ent['doc_node_value'] for ent in annotations['results']['full_matches']]
                results.append({'skills': skills, 'num_skills': len(skills)})
            except Exception as e:
                # Silently handle errors in batch processing
                results.append({'skills': [], 'num_skills': 0})
    return results

def process_parquet_file_in_chunks(filepath, nlp, chunk_size=ROWS_PER_CHUNK):
    """
    Process a single parquet file in chunks.
    
    Parameters
    ----------
    filepath : Path
        Path to the parquet file
    nlp : Pipeline
        SkillNER pipeline
    chunk_size : int
        Number of rows to process at once
    
    Returns
    -------
    pd.DataFrame
        Processed dataframe with extracted skills
    """
    # Read parquet file (only needed columns if specified)
    try:
        pf = pd.read_parquet(filepath)
    except Exception as e:
        print(f"Error reading {filepath.name}: {e}")
        return None
    
    total_rows = len(pf)
    
    # Process in chunks
    all_results = []
    
    for start_idx in range(0, total_rows, chunk_size):
        end_idx = min(start_idx + chunk_size, total_rows)
        
        # Get chunk
        chunk = pf.iloc[start_idx:end_idx].copy()
        
        # Extract skills
        texts = chunk[TEXT_COLUMN].tolist()
        skill_results = process_text_batch(texts, nlp)
        
        # Add results to chunk
        chunk['extracted_skills'] = [r['skills'] for r in skill_results]
        chunk['num_skills'] = [r['num_skills'] for r in skill_results]
        
        all_results.append(chunk)
        
        # Clear memory
        del chunk, texts, skill_results
    
    # Concatenate all chunks
    result_df = pd.concat(all_results, ignore_index=True)
    
    # Clear memory
    del pf, all_results
    gc.collect()
    
    return result_df

def process_single_file(filepath, nlp):
    """
    Process a single file (wrapper for parallel processing).
    
    Parameters
    ----------
    filepath : Path
        Path to the parquet file
    nlp : Pipeline
        SkillNER pipeline
    
    Returns
    -------
    tuple
        (filepath, success, message)
    """
    try:
        # Process file in chunks
        result_df = process_parquet_file_in_chunks(filepath, nlp)
        
        if result_df is None:
            return (filepath, False, "Failed to read file")
        
        # Save result
        output_path = OUTPUT_DIR / f"{filepath.stem}_processed.parquet"
        result_df.to_parquet(output_path, index=False)
        
        # Mark as processed
        mark_file_as_processed(filepath.name)
        
        message = f"Processed {len(result_df)} rows, {result_df['num_skills'].sum()} skills"
        
        # Clear memory
        del result_df
        gc.collect()
        
        return (filepath, True, message)
        
    except Exception as e:
        return (filepath, False, str(e))

def process_file_batch_sequential(files, nlp):
    """Process a batch of files sequentially."""
    for filepath in tqdm(files, desc="Processing files"):
        filepath_obj, success, message = process_single_file(filepath, nlp)
        if success:
            print(f"✓ {filepath.name}: {message}")
        else:
            print(f"✗ {filepath.name}: {message}")

def process_file_batch_parallel(files, nlp, n_workers=N_WORKERS):
    """Process a batch of files in parallel."""
    from joblib import Parallel, delayed
    
    print(f"Processing {len(files)} files using {n_workers} workers...")
    
    # Process in parallel
    results = Parallel(n_jobs=n_workers, verbose=0)(
        delayed(process_single_file)(filepath, nlp)
        for filepath in files
    )
    
    # Report results
    success_count = 0
    for filepath, success, message in results:
        if success:
            print(f"✓ {filepath.name}: {message}")
            success_count += 1
        else:
            print(f"✗ {filepath.name}: {message}")
    
    print(f"\nBatch complete: {success_count}/{len(files)} files processed successfully")

print("Helper functions loaded.")

## Initialize SkillNER Pipeline

In [ ]:
# Initialize the SkillNER pipeline
print("Loading SkillNER pipeline...")
nlp = Pipeline.load("en")
print("✓ Pipeline loaded successfully!")

## Discover Parquet Files

In [ ]:
# Get all parquet files
all_parquet_files = sorted(INPUT_DIR.glob("*.parquet"))
print(f"Found {len(all_parquet_files)} parquet files in {INPUT_DIR}")

if len(all_parquet_files) == 0:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

# Get total size
total_size = sum(f.stat().st_size for f in all_parquet_files)
print(f"Total size: {total_size / (1024**3):.2f} GB")

# Filter out already processed files if resuming
if RESUME_FROM_CHECKPOINT:
    processed_files = get_processed_files()
    files_to_process = [f for f in all_parquet_files if f.name not in processed_files]
    print(f"Already processed: {len(processed_files)} files")
    print(f"Remaining to process: {len(files_to_process)} files")
else:
    files_to_process = all_parquet_files
    print(f"Will process all {len(files_to_process)} files")

# Show first few files
print("\nFirst 5 files to process:")
for f in files_to_process[:5]:
    size_mb = f.stat().st_size / (1024**2)
    print(f"  - {f.name} ({size_mb:.2f} MB)")

## Estimate Processing Time

In [ ]:
# Estimate processing time by processing one file
import time

if len(files_to_process) > 0:
    print("Running speed test with first file...")
    test_file = files_to_process[0]
    
    start_time = time.time()
    test_df = pd.read_parquet(test_file)
    test_rows = len(test_df)
    
    # Process a small sample
    sample_size = min(100, test_rows)
    sample_texts = test_df[TEXT_COLUMN].head(sample_size).tolist()
    sample_results = process_text_batch(sample_texts, nlp)
    
    elapsed = time.time() - start_time
    
    # Estimate
    rows_per_second = sample_size / elapsed
    total_rows_estimate = test_rows * len(files_to_process)
    estimated_seconds = total_rows_estimate / rows_per_second
    
    if USE_PARALLEL:
        estimated_seconds /= N_WORKERS  # Rough parallelization speedup
    
    print(f"\nSpeed test results:")
    print(f"  Processed {sample_size} rows in {elapsed:.2f} seconds")
    print(f"  Speed: {rows_per_second:.2f} rows/second")
    print(f"\nEstimates:")
    print(f"  Total rows to process: ~{total_rows_estimate:,}")
    print(f"  Estimated time: {estimated_seconds/60:.1f} minutes ({estimated_seconds/3600:.2f} hours)")
    if USE_PARALLEL:
        print(f"  (with {N_WORKERS} parallel workers)")
    
    del test_df, sample_texts, sample_results
    gc.collect()

## Process Files in Batches

This is the main processing loop. Files are processed in batches.

In [ ]:
# Process files in batches
total_batches = (len(files_to_process) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\nProcessing {len(files_to_process)} files in {total_batches} batches...")
print(f"Mode: {'PARALLEL' if USE_PARALLEL else 'SEQUENTIAL'}")
print("="*70)

import time
overall_start = time.time()

for batch_idx in range(total_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(files_to_process))
    
    batch_files = files_to_process[start_idx:end_idx]
    
    print(f"\nBatch {batch_idx + 1}/{total_batches}: Processing files {start_idx + 1}-{end_idx}")
    print("-"*70)
    
    batch_start = time.time()
    
    # Process this batch
    if USE_PARALLEL:
        process_file_batch_parallel(batch_files, nlp, n_workers=N_WORKERS)
    else:
        process_file_batch_sequential(batch_files, nlp)
    
    batch_elapsed = time.time() - batch_start
    
    # Force garbage collection after each batch
    gc.collect()
    
    print(f"\n✓ Batch {batch_idx + 1}/{total_batches} completed in {batch_elapsed:.2f} seconds")
    
    # Estimate remaining time
    if batch_idx < total_batches - 1:
        avg_time_per_batch = (time.time() - overall_start) / (batch_idx + 1)
        remaining_batches = total_batches - batch_idx - 1
        estimated_remaining = avg_time_per_batch * remaining_batches
        print(f"  Estimated remaining time: {estimated_remaining/60:.1f} minutes")
    
    print("="*70)

total_elapsed = time.time() - overall_start

print("\n" + "="*70)
print("ALL PROCESSING COMPLETE!")
print("="*70)
print(f"Total time: {total_elapsed/60:.2f} minutes ({total_elapsed/3600:.2f} hours)")
print(f"Average: {total_elapsed/len(files_to_process):.2f} seconds per file")

## Summary Statistics

In [ ]:
# Get all processed output files
output_files = sorted(OUTPUT_DIR.glob("*_processed.parquet"))

print(f"Total output files: {len(output_files)}")

# Calculate comprehensive statistics
total_rows = 0
total_skills = 0
total_size = 0

print("\nCalculating statistics...")
for f in tqdm(output_files, desc="Reading files"):
    df = pd.read_parquet(f, columns=['num_skills'])  # Only read needed column
    total_rows += len(df)
    total_skills += df['num_skills'].sum()
    total_size += f.stat().st_size

avg_skills_per_row = total_skills / total_rows if total_rows > 0 else 0

print("\n" + "="*70)
print("FINAL STATISTICS")
print("="*70)
print(f"Total files processed: {len(output_files)}")
print(f"Total rows: {total_rows:,}")
print(f"Total skills extracted: {total_skills:,}")
print(f"Average skills per row: {avg_skills_per_row:.2f}")
print(f"Total output size: {total_size / (1024**3):.2f} GB")
print(f"Rows with skills: {(total_skills > 0):.0%}" if total_rows > 0 else "N/A")
print("="*70)

## Example: Load and Inspect Results

In [ ]:
# Load one result file as example
if len(output_files) > 0:
    example_file = output_files[0]
    df_example = pd.read_parquet(example_file)
    
    print(f"Example file: {example_file.name}")
    print(f"Shape: {df_example.shape}")
    print(f"\nColumns: {df_example.columns.tolist()}")
    print(f"\nFirst few rows:")
    display(df_example[[TEXT_COLUMN, 'extracted_skills', 'num_skills']].head())
    
    # Show some examples with extracted skills
    df_with_skills = df_example[df_example['num_skills'] > 0]
    if len(df_with_skills) > 0:
        print(f"\nExample rows with extracted skills:")
        for idx, row in df_with_skills.head(3).iterrows():
            print(f"\nRow {idx}:")
            print(f"  Skills ({row['num_skills']}): {row['extracted_skills']}")
else:
    print("No output files found. Please run the processing cells first.")

## Memory Usage Check

In [ ]:
# Check current memory usage
import psutil
import os

process = psutil.Process(os.getpid())
mem_info = process.memory_info()

print("Current Memory Usage:")
print(f"  RSS: {mem_info.rss / (1024**3):.2f} GB")
print(f"  VMS: {mem_info.vms / (1024**3):.2f} GB")

# System memory
vm = psutil.virtual_memory()
print(f"\nSystem Memory:")
print(f"  Total: {vm.total / (1024**3):.2f} GB")
print(f"  Available: {vm.available / (1024**3):.2f} GB")
print(f"  Used: {vm.percent}%")